# Pricing Evaluation Notebook
## Asistente de Venta Reventa por Foto — Evaluación de Calidad, Coste y Fallos

**Requisito**: RDA-002 — Notebook paralelo con metodología, resultados, análisis de coste y conclusiones.

---

### Metodología de evaluación

#### Baseline
Precio fijado por el operador sin ayuda IA, basado en inspección visual del producto y conocimiento del mercado.

#### Propuesta IA v1
Precio generado por `PricingService` usando:
- **LLMClient**: Mock en desarrollo → OpenAI GPT-4o en producción
- **ExternalComparableClient**: Mock en desarrollo → scraping/API de marketplace en producción
- Promedio ponderado de señales internas (historial) y externas (comparables en tiempo real)
- Banda de precio `[min, max]` con `confidence_score` por propuesta

#### Protocolo reproducible (RDA-001)
- Dataset de referencia: exportación de `PublicationDraft` con estado `exported`
- Versión de modelo LLM: fijada en `AIProposal.model_name`
- Versión de prompt: fijada en `AIProposal.prompt_version`
- Métricas calculadas sobre el mismo conjunto de evaluación (sin data leakage)

#### Métricas objetivo (CQR-001)

| Métrica | Descripción | Target v1 |
|---|---|---|
| MAE precio (€) | Error absoluto medio vs precio real de venta | < 15 € |
| Tasa aprobación sin edición | % propuestas aprobadas directamente | > 60% |
| Coste LLM / producto (USD) | Coste por llamada al LLM | < 0.05 USD |
| Latencia p95 E2E (ms) | Tiempo análisis completo por producto | < 8 000 ms |
| Tasa de fallos LLM | % llamadas con error/timeout/bloqueo | < 5% |

In [ ]:
# US1 — Valorar Producto por Foto
# Métricas de calidad y coste del pipeline de análisis IA
# Rellenar con datos reales tras primera iteración piloto

us1_metrics = {
    # Calidad de la propuesta
    "mae_precio_eur": None,          # Error absoluto medio vs precio real de venta
    "tasa_aprobacion_sin_edicion": None,  # % propuestas aprobadas sin modificación
    "delta_precio_medio_eur": None,  # (propuesta - precio_final) medio con signo

    # Coste LLM (extraer de GET /metrics/llm)
    "total_llamadas_llm": None,
    "coste_total_usd": None,
    "coste_por_producto_usd": None,
    "tasa_fallos_llm": None,         # success / (success + timeout + error)
    "latencia_p95_ms": None,

    # Reproducibilidad
    "model_name": "gpt-4o",          # Fijar para comparabilidad
    "prompt_version": "v1.0",
    "n_productos_evaluados": None,
}

print("US1 — métricas pendientes de datos piloto reales")
print({k: v for k, v in us1_metrics.items() if v is not None})

---
## US2 — Flujo de Revisión Humana y Exportación

Mide la calidad operativa del circuito de revisión: cuántas propuestas requieren edición, cuántas se rechazan y cómo evoluciona el delta de precio tras la revisión del operador.

In [ ]:
# US2 — Calidad operativa del flujo de revisión humana
# Extraer de tabla operator_reviews en la base de datos

us2_metrics = {
    # Distribución de decisiones del operador
    "n_revisiones_total": None,
    "pct_aprobadas": None,           # decision == "approve"
    "pct_editadas": None,            # decision == "edit"
    "pct_rechazadas": None,          # decision == "reject"

    # Impacto de las ediciones
    "delta_precio_edicion_medio_eur": None,  # edited_price - suggested_price cuando decision==edit
    "pct_precio_corregido_al_alza": None,
    "pct_precio_corregido_a_la_baja": None,

    # Pipeline de exportación
    "n_borradores_exportados": None,
    "tasa_export_tras_aprobacion": None,     # exported / approved
}

print("US2 — métricas pendientes de datos piloto reales")
print({k: v for k, v in us2_metrics.items() if v is not None})

---
## US3 — Canales de Ingesta Adicionales

Compara el rendimiento del pipeline cuando el producto entra por API directa vs canal adicional normalizado.
Verifica que la política de aprobación humana obligatoria se mantiene en todos los canales.

In [ ]:
# US3 — Comparativa de canales de ingesta API vs canal adicional

us3_metrics = {
    # Canal API directa (source_channel == "api")
    "api_n_productos": None,
    "api_tasa_aprobacion": None,
    "api_coste_por_producto_usd": None,
    "api_latencia_ingesta_p95_ms": None,

    # Canal adicional (source_channel == "whatsapp" | "telegram")
    "canal_adicional_n_productos": None,
    "canal_adicional_tasa_aprobacion": None,
    "canal_adicional_coste_por_producto_usd": None,
    "canal_adicional_latencia_ingesta_p95_ms": None,

    # Verificación de política (debe ser True)
    "aprobacion_humana_obligatoria_canal_adicional": True,  # CQR confirmado por tests
}

# Tabla comparativa
print(f"{'Métrica':<40} {'API directa':>15} {'Canal adicional':>20}")
print("-" * 77)
for k in ["n_productos", "tasa_aprobacion", "coste_por_producto_usd", "latencia_ingesta_p95_ms"]:
    api_val = us3_metrics.get(f"api_{k}")
    canal_val = us3_metrics.get(f"canal_adicional_{k}")
    print(f"{k:<40} {str(api_val):>15} {str(canal_val):>20}")

---
## Análisis de Coste — Guardrail diario (CQR-003)

Verificación del presupuesto diario configurado y proyección de gasto según volumen operativo esperado.

In [ ]:
import os

# Presupuesto diario configurado (CQR-003)
daily_budget_usd = float(os.getenv("LLM_DAILY_BUDGET_USD", "25"))

# Escenarios de volumen
scenarios = {
    "Piloto (50 prod/día)":    50,
    "Normal (150 prod/día)":  150,
    "Pico (300 prod/día)":    300,
    "Máximo (500 prod/día)":  500,
}

coste_por_producto_usd = us1_metrics.get("coste_por_producto_usd") or 0.05  # estimado si no hay datos reales

print(f"Presupuesto diario: {daily_budget_usd} USD")
print(f"Coste estimado/producto: {coste_por_producto_usd} USD\n")
print(f"{'Escenario':<30} {'Coste/día (USD)':>18} {'Dentro presupuesto':>20}")
print("-" * 70)
for scenario, volumen in scenarios.items():
    coste_dia = volumen * coste_por_producto_usd
    ok = "✓" if coste_dia <= daily_budget_usd else "✗ EXCEDE"
    print(f"{scenario:<30} {coste_dia:>18.2f} {ok:>20}")

---
## Conclusiones y Comparativa Final — Baseline vs Propuesta IA

**Completar tras primera iteración piloto real** con los datos de `us1_metrics` y `us2_metrics`.

| Métrica | Baseline (manual) | Propuesta IA v1 | Mejora |
|---|---|---|---|
| MAE precio (€) | — | — | — |
| Tasa aprobación sin edición | — | — | — |
| Coste por producto (USD) | 0 | — | — |
| Latencia análisis p95 (ms) | — (manual) | — | — |
| Tasa fallos | 0% | — | — |

### Criterios de parada (CQR-003)
- Corte automático al superar presupuesto diario (`LLM_DAILY_BUDGET_USD`)
- Alerta si `tasa_fallos_llm` > 10% en ventana de 1 hora
- Revisión manual si `tasa_aprobacion_sin_edicion` < 40% durante 2 días consecutivos

In [ ]:
# Comparativa final baseline vs propuesta IA (CQR-002)
# Ejecutar una vez disponibles us1_metrics con datos reales

baseline = {
    "mae_precio_eur": None,
    "tasa_aprobacion_sin_edicion": None,  # referencia: estimación histórica operadores
    "coste_por_producto_usd": 0.0,
    "tasa_fallos": 0.0,
}

propuesta_ia = {
    "mae_precio_eur": us1_metrics["mae_precio_eur"],
    "tasa_aprobacion_sin_edicion": us1_metrics["tasa_aprobacion_sin_edicion"],
    "coste_por_producto_usd": us1_metrics["coste_por_producto_usd"],
    "tasa_fallos": us1_metrics["tasa_fallos_llm"],
}

def mejora(baseline_val, propuesta_val, higher_is_better=False):
    if baseline_val is None or propuesta_val is None:
        return "pendiente"
    delta = propuesta_val - baseline_val
    if higher_is_better:
        return f"+{delta:.1f}" if delta >= 0 else f"{delta:.1f}"
    return f"{delta:.1f}" if delta >= 0 else f"{delta:.1f}"

print(f"{'Métrica':<35} {'Baseline':>12} {'IA v1':>12} {'Delta':>12}")
print("-" * 73)
print(f"{'MAE precio (€)':<35} {str(baseline['mae_precio_eur']):>12} {str(propuesta_ia['mae_precio_eur']):>12} {mejora(baseline['mae_precio_eur'], propuesta_ia['mae_precio_eur']):>12}")
print(f"{'Tasa aprobación sin edición':<35} {str(baseline['tasa_aprobacion_sin_edicion']):>12} {str(propuesta_ia['tasa_aprobacion_sin_edicion']):>12} {mejora(baseline['tasa_aprobacion_sin_edicion'], propuesta_ia['tasa_aprobacion_sin_edicion'], True):>12}")
print(f"{'Coste/producto (USD)':<35} {str(baseline['coste_por_producto_usd']):>12} {str(propuesta_ia['coste_por_producto_usd']):>12} {'—':>12}")
print(f"{'Tasa fallos LLM':<35} {str(baseline['tasa_fallos']):>12} {str(propuesta_ia['tasa_fallos']):>12} {'—':>12}")